In [147]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [148]:
# Import modules
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from lets_plot import *

LetsPlot.setup_html()

In [149]:
# Load in data
housing = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')

# Other data
# https://simplemaps.com/city/seattle/zips/income-household-median
median_income_2024 = pd.read_csv('average-income.csv')
median_house_price_2024 = pd.read_csv('median-house-price.csv')

In [150]:
housing.head()

,id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
0,1565930130,20141104T000000,4,3.25,3760,4675,2.0,0,0,3,...,2740,1020,2007,0,98038,47.3862,-122.048,3280,4033,429900.0
1,3279000420,20150115T000000,3,1.75,1460,7800,1.0,0,0,2,...,1040,420,1979,0,98023,47.3035,-122.382,1310,7865,233000.0
2,194000575,20141014T000000,4,1.00,1340,5800,1.5,0,2,3,...,1340,0,1914,0,98116,47.5658,-122.389,1900,5800,455000.0
3,2115510160,20141208T000000,3,1.75,1440,8050,1.0,0,0,3,...,1440,0,1985,0,98023,47.3187,-122.390,1790,7488,258950.0
4,7522500005,20140815T000000,2,1.50,1780,4750,1.0,0,0,4,...,1080,700,1947,0,98117,47.6859,-122.395,1690,5962,555000.0


In [151]:
median_house_price = housing.groupby('zipcode')['price'].median().reset_index(name='median_house_price')
median_house_price.head()

,zipcode,median_house_price
0,98001,260000.0
1,98002,235000.0
2,98003,268000.0
3,98004,1162500.0
4,98005,760000.0


In [178]:
'''
Similarly, look at sqft_living and sqft_lot sitting side by side. A 2,000 sqft house on a 2,500 sqft lot is a very 
different property than the same house on a 20,000 sqft lot. What single number could capture that relationship?
'''

# Other ideas for cleaning data:
# - change the 33 bedroom house to 3 bedrooms

# Get average housing price by zipcode?

def clean_data(df):
    df = df[df['bedrooms'] != 0]
    df = df[df['bathrooms'] != 0]

    df = df.assign(
        was_rennovated = df['yr_renovated'] != 0,
        age = df['date'].str[:4].astype(int) - df[['yr_built', 'yr_renovated']].max(axis=1),
        house_percentage = df['sqft_living'] / df['sqft_lot']
    )

    # Median income by zipcode
    df = pd.merge(df, median_income_2024, on='zipcode', how='inner')

    # Median house price by zipcode
    df = pd.merge(df, median_house_price, on='zipcode', how='inner')
    df = pd.merge(df, median_house_price_2024, on='zipcode', how='inner')

    # Drop these columns
    df = df.drop(columns=['id', 'date', 'yr_renovated'])

    return df

clean_df = clean_data(housing)

clean_df.head()

,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,...,long,sqft_living15,sqft_lot15,price,was_rennovated,age,house_percentage,median_income_2024,median_house_price,median_house_price_2024
0,4,1.00,1340,5800,1.5,0,2,3,7,1340,...,-122.389,1900,5800,455000.0,False,100,0.231034,126151,566500.0,976323
1,2,1.50,1780,4750,1.0,0,0,4,7,1080,...,-122.395,1690,5962,555000.0,False,67,0.374737,182984,545000.0,1042495
2,4,3.50,2650,3060,2.0,0,0,3,9,2060,...,-122.332,1470,3060,1008000.0,False,13,0.866013,131154,552000.0,1046407
3,3,2.25,1670,3135,2.0,0,0,3,8,1220,...,-122.396,1630,1596,540000.0,False,13,0.532695,180789,689800.0,1200274
4,2,1.00,870,7500,1.0,0,0,3,7,870,...,-122.384,1240,5709,379950.0,False,67,0.116000,149269,485000.0,925851


In [210]:
# Prepare data for ML
X = clean_df.drop(columns=['price'])
y = clean_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [211]:
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse'
}

model = XGBRegressor(**params)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
predictions

array([254042.56, 447840.06, 594377.94, ..., 435791.97, 782542.3 ,
       410441.88], shape=(1392,), dtype=float32)

In [212]:
r2 = model.score(X_test, y_test)
rmse = root_mean_squared_error(y_test, predictions)

print(f'R2: {r2}')
print(f'RMSE: {rmse}')

R2: 0.8720259635370521
RMSE: 117865.70390271711


In [208]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=True)

(
    ggplot(importance_df, aes(x='Importance', y='Feature')) +
    geom_bar(stat='identity', fill='steelblue') +
    labs(
        title='XGBRegressor Feature Importance'
    )
)

In [209]:
mini_df = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing_holdout_test_mini.csv')
mini_df = clean_data(mini_df)
mini_preds = model.predict(mini_df)
my_predictions = pd.DataFrame(mini_preds, columns=['price'])
my_predictions.to_csv('ctrl-alt-elite-module3-predictions.csv', index=False)